# Waypoint-6m fine-tuning webinar — classification

An end-to-end walkthrough of the `waypoint` library on **Compass task 8** (`roswall`): embed microbiome samples with the pretrained model, fine-tune it on infant delivery mode (binary), and see how the embedding geometry shifts.

Every heavy step (data download, embedding, fine-tuning) is a `waypoint` CLI call and its output caches to `artifacts/task8_birth_mode/`. The default is `USE_CACHE = True` — on the first run each step computes and caches; on subsequent runs the cache is loaded and the whole notebook finishes in seconds. Set `USE_CACHE = False` to force a rerun.

The parallel regression demo is in [`webinar_finetune_regression.ipynb`](webinar_finetune_regression.ipynb). Both notebooks share [`finetune.yaml`](finetune.yaml) and [`webinar_utils.py`](webinar_utils.py).

## Setup

Imports (from the shared [`webinar_utils.py`](webinar_utils.py) module), tunable knobs (`USE_CACHE`, `TASK` dict with target/covariate), and the artifact directory setup.

In [ ]:
from __future__ import annotations

import json
import time
from pathlib import Path

import pandas as pd
import plotly.express as px
from datasets import load_dataset
from plotly.subplots import make_subplots
from webinar_utils import (
    COMPASS_REPO, SEED,
    DELIVERY_MODE_COLORS,
    paths_for, load_embeddings,
    project, logreg_score,
    plot_classification_stage, plot_classification_stage_3d,
)

# --- knobs -----------------------------------------------------------------
# USE_CACHE=True   →  each step loads its cached artifact if present, else computes and caches.
# USE_CACHE=False  →  each step recomputes from scratch and overwrites the cache.
# Default is True so that after the first (slow) run, the webinar plays back instantly.
USE_CACHE = True

WEBINAR_DIR = Path.cwd()
FINETUNE_CONFIG = WEBINAR_DIR / "finetune.yaml"
ARTIFACT_DIR = WEBINAR_DIR / "artifacts"
ARTIFACT_DIR.mkdir(exist_ok=True)

# Task config for Compass task 8 (binary classification of infant delivery mode).
TASK = {
    "hub_config": "roswall",
    "target": "Delivery Mode",
    "covariate": None,
    "task_type": "classification",
    "tag": "task8_birth_mode",
    "config": FINETUNE_CONFIG,
}

PATHS = paths_for(ARTIFACT_DIR, TASK["tag"])

# Persisted between cells — accumulates wall-clock timings and money-slide metrics.
timings: dict[str, float] = (
    json.loads(PATHS["timings"].read_text()) if USE_CACHE and PATHS["timings"].exists() else {}
)
metrics: dict[str, dict] = (
    json.loads(PATHS["metrics"].read_text()) if USE_CACHE and PATHS["metrics"].exists() else {}
)
print(f"USE_CACHE={USE_CACHE}   |   waypoint CLI auto-detects device (cuda/mps/cpu)")
print(f"Webinar dir: {WEBINAR_DIR}")
print(f"Artifact dir for this task: {PATHS['samples'].parent}")
print(f"Config: {FINETUNE_CONFIG.name}  {'✓' if FINETUNE_CONFIG.exists() else '(missing!)'}")

/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


USE_CACHE=True   |   waypoint CLI auto-detects device (cuda/mps/cpu)
Webinar dir: /Users/neythen/Desktop/repos/waypoint/examples/webinar
Artifact dir for this task: /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode
Config: finetune.yaml  ✓


## 1. Load Compass task 8 (`roswall`)

The `roswall` config is a longitudinal study of infant gut microbiomes; the target is whether the infant was born vaginally or via c-section.

Every row is one sample in the **waypoint format** — the shape every `waypoint` CLI command expects:

| column | type | purpose |
|---|---|---|
| `Taxa` | `list[str]` | taxonomic labels present in the sample |
| `Relative Abundances` | `list[float]` | matching proportions, sum to ~1 |
| `Delivery Mode` | `str` | classification target (e.g. `"Vaginal"` / `"Cesarean"`) |

No covariate column — the model predicts the class directly from the microbiome composition. The cell below writes `artifacts/task8_birth_mode/samples.parquet` — the `--data` input for every CLI call.

In [2]:
if USE_CACHE and PATHS["samples"].exists():
    df = pd.read_parquet(PATHS["samples"])
    print(f"Loaded cached samples from {PATHS['samples']}")
else:
    # Pull roswall from Compass and pool splits.
    ds = load_dataset(COMPASS_REPO, TASK["hub_config"])
    frames = [ds[s].to_pandas() for s in ("train", "validation", "test") if s in ds]
    df = pd.concat(frames, ignore_index=True)
    # astype(str) turns missing values into the literal "nan"; drop those.
    df[TASK["target"]] = df[TASK["target"]].astype(str)
    df = df[df[TASK["target"]].notna() & (df[TASK["target"]] != "nan")].reset_index(drop=True)
    # Only three columns for classification (no covariate).
    df[["Taxa", "Relative Abundances", TASK["target"]]].to_parquet(PATHS["samples"])
    print(f"Downloaded and cached {len(df):,} samples to {PATHS['samples']}")

counts = df[TASK["target"]].value_counts()
print(f"{len(df):,} samples   |   classes: {counts.to_dict()}")
df[[TASK["target"]]].head(5)

Downloaded and cached 2,031 samples to /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/samples.parquet
2,031 samples   |   classes: {'Vaginal': 1279, 'C-section': 752}


,Delivery Mode
0,C-section
1,C-section
2,C-section
3,C-section
4,C-section


In [3]:
# Class balance. Heavy imbalance means "always predict majority" already has high
# accuracy, so watch macro-F1 and ROC-AUC instead of raw accuracy in the money slide.
counts_df = counts.rename_axis(TASK["target"]).reset_index(name="n")
px.bar(counts_df, x=TASK["target"], y="n",
       title=f"Class distribution: {TASK['target']}", height=320)

## 2. Embed with the base Waypoint-6m

Runs `waypoint embed` in a subshell: loads the pretrained model from HF Hub, tokenizes each sample, forward-passes, pools, and writes one row per sample to a parquet (`dim_0 … dim_N` columns).

```bash
waypoint embed \
    --model outpost-bio/Waypoint-6m \
    --data artifacts/task8_birth_mode/samples.parquet \
    --output artifacts/task8_birth_mode/base_embeddings.parquet \
    --pooling last_token --batch_size 32 --max_length 512
```

In [4]:
if USE_CACHE and PATHS["base_emb"].exists():
    print(f"Loaded cached base embeddings from {PATHS['base_emb']}")
else:
    t0 = time.time()
    !waypoint embed \
        --model {BASE_MODEL_ID} \
        --data "{PATHS['samples']}" \
        --output "{PATHS['base_emb']}" \
        --pooling {POOLING} \
        --batch_size {BATCH_SIZE} \
        --max_length {MAX_LENGTH}
    timings[f"{TASK['tag']}__embed_base_s"] = round(time.time() - t0, 2)

base_emb = load_embeddings(PATHS["base_emb"])
print(f"Shape: {base_emb.shape}")

Loading waypoint-format data from /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/samples.parquet ...
Loaded 2,031 samples
Loading model from outpost-bio/Waypoint-6m ...
Loading weights: 100%|██████████████████████| 100/100 [00:00<00:00, 9393.95it/s]
Using device: mps
Loaded token_std_means (14389 tokens)
Tokenising ...
Generating embeddings ...
Embedding: 100%|█████████████████████████████| 64/64 [00:07<00:00,  8.32batch/s]
Saved embeddings of shape (2031, 256) to /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/base_embeddings.parquet
Shape: (2031, 256)


## 3. Visualize base embeddings

PCA + t-SNE colored by delivery mode (blue = Vaginal, red = C-section). Before fine-tuning we expect the two classes to be intermixed — the pretrained model doesn't yet know this target.

In [5]:
base_proj = project(base_emb, seed=SEED)
plot_classification_stage(base_proj, df, TASK["target"],
                          "Base Waypoint-6m embeddings (task 8)",
                          color_map=DELIVERY_MODE_COLORS)

/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath

In [6]:
# Same projections in 3D — rotate to see whether the classes separate along an axis 2D flattens.
plot_classification_stage_3d(base_proj, df, TASK["target"],
                             "Base Waypoint-6m — 3D embeddings (task 8)",
                             color_map=DELIVERY_MODE_COLORS)

## 4. Baseline: logistic regression on pretrained embeddings

Class-balanced logistic regression on base pretrained embeddings, 80/20 random split. Reports accuracy, macro-F1, and (for binary) ROC-AUC. This is the "no fine-tuning" floor for comparison in section 8.

In [7]:
y = df[TASK["target"]].to_numpy()
base_logreg = logreg_score(base_emb, y)
msg = (f"Base embeddings → Accuracy = {base_logreg['accuracy']:.3f}   "
       f"macro-F1 = {base_logreg['f1_macro']:.3f}")
if "roc_auc" in base_logreg:
    msg += f"   ROC-AUC = {base_logreg['roc_auc']:.3f}"
print(msg)

Base embeddings → Accuracy = 0.714   macro-F1 = 0.691   ROC-AUC = 0.748


/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: over

## 5. Fine-tune the model (classification)

Runs `waypoint finetune` with `task_type=classification`, no covariate. Uses the shared [`finetune.yaml`](finetune.yaml) config.

```bash
waypoint finetune \
    --model outpost-bio/Waypoint-6m \
    --data artifacts/task8_birth_mode/samples.parquet \
    --output_dir artifacts/task8_birth_mode/finetune_run \
    --task_type classification \
    --target "Delivery Mode" \
    --config finetune.yaml
```

With `USE_CACHE=True` (default), if the checkpoint already exists on disk from a previous run, this cell just prints the path and moves on — no retraining.

In [8]:
if USE_CACHE and PATHS["ft_model"].exists():
    print(f"Using cached fine-tuned checkpoint at {PATHS['ft_model']}")
else:
    PATHS["ft_dir"].mkdir(parents=True, exist_ok=True)
    t0 = time.time()
    !waypoint finetune \
        --model {BASE_MODEL_ID} \
        --data "{PATHS['samples']}" \
        --output_dir "{PATHS['ft_dir']}" \
        --task_type {TASK['task_type']} \
        --target "{TASK['target']}" \
        --config "{TASK['config']}"
    key = f"{TASK['tag']}__finetune_s"
    timings[key] = round(time.time() - t0, 2)
    print(f"Fine-tune finished in {timings[key]}s")

if not PATHS["ft_model"].exists():
    raise FileNotFoundError(f"Fine-tune did not produce {PATHS['ft_model']}")

Loading data from /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/samples.parquet ...
Split sizes: 1421 train / 305 validation / 305 test
Loading model from outpost-bio/Waypoint-6m ...
Loading weights: 100%|█████████████████████| 100/100 [00:00<00:00, 26532.79it/s]
trainable params: 180,224 || all params: 10,314,752 || trainable%: 1.7472
Label maps: {'Delivery Mode': {'C-section': 0, 'Vaginal': 1}}
Fine-tuning ...
  0%|                                                  | 0/4500 [00:00<?, ?it/s]/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/torch/utils/data/dataloader.py:752: UserWarning: 'pin_memory' argument is set as true but not supported on MPS now, device pinned memory won't be used.
  super().__init__(loader)
{'loss': '0.7048', 'grad_norm': '0.7595', 'learning_rate': '1.2e-07', 'epoch': '0.1111'}
{'loss': '0.6954', 'grad_norm': '0.8002', 'learning_rate': '2.7e-07', 'epoch': '0.2222'}
{'loss': '0.7139', 'grad_norm': '0.794

## 6. Embed with the fine-tuned checkpoint

Same `waypoint embed` command as section 2, pointing at the fine-tuned `best_model/`.

```bash
waypoint embed \
    --model artifacts/task8_birth_mode/finetune_run/best_model \
    --data artifacts/task8_birth_mode/samples.parquet \
    --output artifacts/task8_birth_mode/finetuned_embeddings.parquet \
    --pooling last_token --batch_size 32 --max_length 512
```

In [9]:
if USE_CACHE and PATHS["ft_emb"].exists():
    print(f"Loaded cached fine-tuned embeddings from {PATHS['ft_emb']}")
else:
    t0 = time.time()
    !waypoint embed \
        --model "{PATHS['ft_model']}" \
        --data "{PATHS['samples']}" \
        --output "{PATHS['ft_emb']}" \
        --pooling {POOLING} \
        --batch_size {BATCH_SIZE} \
        --max_length {MAX_LENGTH}
    timings[f"{TASK['tag']}__embed_ft_s"] = round(time.time() - t0, 2)

ft_emb = load_embeddings(PATHS["ft_emb"])
print(f"Shape: {ft_emb.shape}")

Loading waypoint-format data from /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/samples.parquet ...
Loaded 2,031 samples
Loading model from /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/finetune_run/best_model ...
Loading weights: 100%|█████████████████████| 100/100 [00:00<00:00, 42829.61it/s]
Using device: mps
Loaded token_std_means (14389 tokens)
Tokenising ...
Generating embeddings ...
Embedding: 100%|█████████████████████████████| 64/64 [00:07<00:00,  8.08batch/s]
Saved embeddings of shape (2031, 256) to /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/finetuned_embeddings.parquet
Shape: (2031, 256)


## 7. Visualize fine-tuned embeddings

Same layout as section 3. If fine-tuning worked, the two delivery-mode classes should visibly cluster apart — especially in t-SNE.

In [10]:
ft_proj = project(ft_emb, seed=SEED)
plot_classification_stage(ft_proj, df, TASK["target"],
                          "Fine-tuned Waypoint-6m embeddings (task 8)",
                          color_map=DELIVERY_MODE_COLORS)

/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:350: RuntimeWarning: invalid value encountered in matmul
  Q, _ = normalizer(A @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: divide by zero encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath.py:351: RuntimeWarning: overflow encountered in matmul
  Q, _ = normalizer(A.T @ Q)
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/utils/extmath

In [11]:
# Same 3D projections, on the fine-tuned classification embeddings.
plot_classification_stage_3d(ft_proj, df, TASK["target"],
                             "Fine-tuned Waypoint-6m — 3D embeddings (task 8)",
                             color_map=DELIVERY_MODE_COLORS)

## 8. Baseline vs fine-tuned — classification money slide

Logistic regression on frozen embeddings from both stages. The delta measures how much fine-tuning improved linear separability of the classes.

In [12]:
ft_logreg = logreg_score(ft_emb, y)

cols = {
    "stage": ["base Waypoint-6m", "fine-tuned Waypoint-6m"],
    "Accuracy": [base_logreg["accuracy"], ft_logreg["accuracy"]],
    "macro-F1": [base_logreg["f1_macro"], ft_logreg["f1_macro"]],
}
if "roc_auc" in base_logreg and "roc_auc" in ft_logreg:
    cols["ROC-AUC"] = [base_logreg["roc_auc"], ft_logreg["roc_auc"]]
summary = pd.DataFrame(cols)
metrics[TASK["tag"]] = {
    "target": TASK["target"],
    "n_samples": int(len(df)),
    "classes": base_logreg["classes"],
    "base_logreg":      {k: base_logreg[k] for k in base_logreg if k not in ("y_true", "y_pred", "classes")},
    "finetuned_logreg": {k: ft_logreg[k]   for k in ft_logreg   if k not in ("y_true", "y_pred", "classes")},
}
summary.style.format({c: "{:.3f}" for c in summary.columns if c != "stage"})

/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: divide by zero encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: overflow encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:200: RuntimeWarning: invalid value encountered in matmul
  raw_prediction = X @ weights + intercept
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: divide by zero encountered in matmul
  grad[:n_features] = X.T @ grad_pointwise + l2_reg_strength * weights
/Users/neythen/Desktop/repos/waypoint/.venv/lib/python3.10/site-packages/sklearn/linear_model/_linear_loss.py:330: RuntimeWarning: over

,stage,Accuracy,macro-F1,ROC-AUC
0,base Waypoint-6m,0.714,0.691,0.748
1,fine-tuned Waypoint-6m,0.761,0.742,0.773


In [13]:
# Confusion matrices side-by-side. Diagonal = correct; off-diagonal = class swap.
# Fine-tuning should shrink the off-diagonal counts noticeably.
from sklearn.metrics import confusion_matrix

classes = base_logreg["classes"]
cm_base = confusion_matrix(base_logreg["y_true"], base_logreg["y_pred"], labels=classes)
cm_ft   = confusion_matrix(ft_logreg["y_true"],   ft_logreg["y_pred"],   labels=classes)

fig = make_subplots(rows=1, cols=2, subplot_titles=("base", "fine-tuned"))
for cm, col_idx in ((cm_base, 1), (cm_ft, 2)):
    fig.add_trace(
        px.imshow(cm, x=classes, y=classes, text_auto=True,
                  color_continuous_scale="Blues").data[0],
        row=1, col=col_idx,
    )
    fig.update_xaxes(title_text="predicted", row=1, col=col_idx)
    fig.update_yaxes(title_text="true",      row=1, col=col_idx)
fig.update_layout(title="Task 8 — Confusion matrices (held-out test set)", height=420,
                  coloraxis_showscale=False)
fig

## Persist metrics + timings

Writes `metrics.json` (money-slide numbers) and `timings.json` (wall-clock per step) into this task's `artifacts/` subdirectory. Handy for pulling into a slide deck or comparing runs later.

In [14]:
PATHS["metrics"].write_text(json.dumps(metrics, indent=2))
PATHS["timings"].write_text(json.dumps(timings, indent=2))
print(f"Wrote {PATHS['metrics']}")
print(f"Wrote {PATHS['timings']}")
print(json.dumps({"metrics": metrics, "timings": timings}, indent=2))

Wrote /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/metrics.json
Wrote /Users/neythen/Desktop/repos/waypoint/examples/webinar/artifacts/task8_birth_mode/timings.json
{
  "metrics": {
    "task8_birth_mode": {
      "target": "Delivery Mode",
      "n_samples": 2031,
      "classes": [
        "C-section",
        "Vaginal"
      ],
      "base_logreg": {
        "accuracy": 0.7142857142857143,
        "f1_macro": 0.6907519764662622,
        "roc_auc": 0.7478906249999999
      },
      "finetuned_logreg": {
        "accuracy": 0.7610837438423645,
        "f1_macro": 0.7417827754647084,
        "roc_auc": 0.7730989583333332
      }
    }
  },
  "timings": {
    "task8_birth_mode__embed_base_s": 12.02,
    "task8_birth_mode__finetune_s": 2235.45,
    "task8_birth_mode__embed_ft_s": 12.1
  }
}


## Bring your own data

Two shapes of parquet input, no other changes required.

**Already have taxa + abundance lists?** Save a parquet in waypoint format:

```python
df.to_parquet("my_data.parquet")   # columns: Taxa (list[str]), Relative Abundances (list[float]), <target>
```

**Have a raw abundance matrix (taxa × samples TSV + sample metadata CSV)?** Convert first:

```bash
waypoint prepare-dataset --input matrix.tsv --output my_data.parquet --metadata labels.csv
```

Then run the same three CLI commands this notebook uses. Copy `finetune.yaml` next to your data and tweak `num_epochs` / `learning_rate` / `warmup_steps` for your dataset size:

```bash
waypoint embed    --model outpost-bio/Waypoint-6m --data my_data.parquet --output my_embeddings.parquet
waypoint finetune --model outpost-bio/Waypoint-6m --data my_data.parquet \
                  --output_dir outputs/my_finetune \
                  --task_type classification --target "My Target" \
                  --config finetune.yaml
waypoint embed    --model outputs/my_finetune/best_model --data my_data.parquet --output my_ft_embeddings.parquet
```

Flip `use_lora: true` in the YAML if you need parameter-efficient tuning to fit a smaller GPU.